# Slider 2v2 AI 학습 (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Perdume/Slider/blob/master/colab/slider_2v2_training.ipynb)

말 2개씩 두는 Slider 의 **평가 함수 가중치**를 자가대전 데이터로 튜닝합니다 (Texel 방식).

1. 레포를 받아 Java 코드를 컴파일합니다.
2. 현재 가중치로 AI 끼리 자가대전을 두고, 국면마다 *평가 특징 · 탐색 점수 · 최종 결과(승/무/패)* 를 JSONL 로 기록합니다.
3. 특징으로 결과를 예측하는 로지스틱 모델을 맞춰 새 가중치를 계산합니다 (`P(승) = sigmoid(점수 / 200)`).
4. **새 가중치 vs 현재 가중치** 대국으로 정말 강해졌는지 확인하고, 이기면 채택합니다.
5. 1~4를 `ROUNDS` 번 반복하고 `weights.json` 을 내려받습니다.

GPU 는 필요 없습니다. 기본 설정은 CPU 2코어 기준 라운드당 약 20분(자가대전 15분 + 대국 확인 5분)입니다. 위에서부터 순서대로 실행하세요.

> 이 방식이 통하려면 "평가 점수가 승패를 얼마나 설명하는지"가 데이터에서 보여야 합니다. 5번 단계가 그 신호의 세기를 직접 출력하니, 신호가 약하면 판 수를 늘리거나(`GAMES_PER_ROUND`) 수당 시간을 늘려(`TIME_MS`) 보세요.

## 1. 설정

In [ ]:
import os, sys, json, shutil, subprocess, pathlib
import numpy as np
import pandas as pd

# ===== 레포 =====
REPO_URL = "https://github.com/Perdume/Slider.git"
BRANCH = "master"

# ===== 학습 설정 (처음엔 그대로 두고, 시간이 남으면 늘려 보세요) =====
ROUNDS = 2             # "자가대전 -> 학습 -> 대국 확인" 반복 횟수
GAMES_PER_ROUND = 600  # 라운드마다 자가대전 판 수 (한 판에 국면 약 100개). 2코어 기준 600판 ~ 15분
TIME_MS = 30           # 자가대전에서 수당 탐색 시간(ms). 길수록 정확하지만 느립니다
RANDOM_PLIES = 4       # 판 시작의 무작위 수 (시작 국면 다양화)
EPSILON = 0.05         # 그 뒤에도 이 확률로 무작위 수를 둠 (데이터 다양화)
MATCH_GAMES = 100      # 라운드 끝에 새 가중치 vs 현재 가중치 대국 수 (짝수)
MATCH_TIME_MS = 50
ACCEPT_SCORE = 0.55    # 새 가중치의 대국 점수가 이 값 이상이면 채택 (0.5 = 동률). 시간 제한 탐색이라 같은 가중치끼리도 0.5 근처에서 흔들립니다
SEED = 1

THREADS = os.cpu_count() or 2
WORK_DIR = pathlib.Path(os.environ.get("SLIDER_WORK_DIR", "/content/slider_work"))
DATA_DIR = WORK_DIR / "data"
BUILD_DIR = WORK_DIR / "build"
for d in (WORK_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("work dir:", WORK_DIR, "| threads:", THREADS)

## 2. Java 준비 · 코드 받기 · 컴파일

In [ ]:
def sh(cmd):
    print("$", " ".join(map(str, cmd)) if isinstance(cmd, list) else cmd)
    subprocess.run(cmd, shell=isinstance(cmd, str), check=True)

if shutil.which("javac") is None:
    sh("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless")

# SLIDER_REPO_DIR 환경변수로 이미 받아 둔 레포를 쓸 수도 있습니다 (로컬 실행용)
if os.environ.get("SLIDER_REPO_DIR"):
    REPO_DIR = pathlib.Path(os.environ["SLIDER_REPO_DIR"])
else:
    REPO_DIR = WORK_DIR / "Slider"
    if REPO_DIR.exists():
        sh(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    else:
        sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])

needed = REPO_DIR / "src" / "main" / "java" / "org" / "example" / "Learning" / "SelfPlay.java"
if not needed.exists():
    raise RuntimeError(
        "받아온 레포에 2v2 학습 코드(SelfPlay.java)가 없습니다. 2v2 커밋이 GitHub " + BRANCH + " 브랜치에 "
        "푸시되지 않았거나 예전 버전을 받은 것입니다. 푸시한 뒤 WORK_DIR 의 Slider 폴더를 지우고 다시 실행하세요.")

shutil.rmtree(BUILD_DIR, ignore_errors=True)
BUILD_DIR.mkdir(parents=True)
sources = [str(p) for p in (REPO_DIR / "src" / "main" / "java").rglob("*.java")]
sh(["javac", "-encoding", "UTF-8", "-d", BUILD_DIR, *sources])
print("컴파일 완료:", len(sources), "개 파일")

## 3. 자가대전 · 대국 실행 함수

- `SelfPlay`: 같은 가중치의 AI 끼리 두며 학습 데이터(JSONL)를 만듭니다.
- `Match`: 가중치 두 벌을 맞붙입니다. 시작 국면마다 쪽을 바꿔 두 판씩 둬서 선공/후공 차이를 없앱니다.

In [ ]:
def run_java(main_class, args):
    cmd = ["java", "-Dfile.encoding=UTF-8", "-cp", str(BUILD_DIR), main_class, *map(str, args)]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8")
    lines = []
    for line in proc.stdout:
        line = line.rstrip("\n")
        lines.append(line)
        if not line.startswith("RESULT ") and not line.startswith("Picked up"):
            print("   ", line)
    if proc.wait() != 0:
        raise RuntimeError(f"{main_class} 실행 실패 (exit {proc.returncode})")
    return lines


def run_selfplay(weights_path, out_path, games, seed):
    run_java("org.example.Learning.SelfPlay", [
        "--games", games, "--time-ms", TIME_MS, "--random-plies", RANDOM_PLIES, "--epsilon", EPSILON,
        "--threads", THREADS, "--seed", seed, "--weights", weights_path, "--out", out_path,
    ])


def run_match(a_path, b_path, seed):
    """a 가중치 vs b 가중치. {'aWins','draws','bWins','score','low','high'} (score 는 a 의 점수)"""
    lines = run_java("org.example.Learning.Match", [
        "--a", a_path, "--b", b_path, "--games", MATCH_GAMES, "--time-ms", MATCH_TIME_MS,
        "--threads", THREADS, "--seed", seed,
    ])
    for line in lines:
        if line.startswith("RESULT "):
            return json.loads(line[len("RESULT "):])
    raise RuntimeError("Match 결과를 찾지 못했습니다")

## 4. 학습 코드 (Texel 튜닝)

평가 점수는 `가중치 · 특징` 의 선형식이고, 승리 확률을 `sigmoid(점수 / K)` 로 봅니다.
자가대전 결과(승 1 / 무 0.5 / 패 0)와의 제곱오차를 최소화하는 가중치를 L-BFGS 로 찾습니다.

- **눈금 K 를 먼저 보정**: 점수가 승률로 얼마나 잘 이어지는지(`K`)를 *현재 가중치를 고정한 채* 먼저 맞춥니다. 이걸 건너뛰면 "점수가 과하게 크다"는 정보를 가중치 전체를 줄이는 것으로 떠안아서, 탐색 상수(`MARGIN` 등)와 점수 크기가 어긋납니다. 그래서 `K` 는 고정하고 가중치는 *형태*만 바꿉니다.
- **정규화**: 손으로 정한 기본 가중치에서 너무 멀어지지 않게 `lambda * |(w - w0) / 200|^2` 을 더합니다. 데이터가 적을수록 중요합니다.
- **검증**: 판(game) 단위로 80/20 으로 나눠서, 정규화 세기 `lambda` 를 검증 오차가 가장 작은 값으로 고릅니다. 같은 판의 국면이 학습/검증에 섞이면 결과가 부풀려지므로 판 단위로 나눕니다.
- **걸러내는 국면**: 탐색이 이미 승패를 확정한 국면(|점수| ≥ 50000)은 평가 함수의 일이 아니라서 뺍니다.

In [ ]:
from scipy.optimize import minimize

# EvalWeights.java 의 NAMES / 기본값과 같아야 합니다 (데이터를 읽을 때 길이를 확인합니다)
NAMES = ["threat1", "threat2", "threat2PerThreat", "mobility", "reach",
         "plan2", "plan3", "plan4", "plan5", "position", "tempo"]
DEFAULT_WEIGHTS = dict(zip(NAMES, [-150, -400, -50, 1, 2, 150, 70, 30, 12, 1, 0]))
FORCED_SCORE = 50_000  # 이 이상이면 탐색이 승패를 확정한 국면
LAMBDAS = [0.0, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
REG_UNIT = 200.0       # 정규화에서 가중치 차이를 재는 단위
MAX_ABS_WEIGHT = 3000


def write_weights(path, w):
    with open(path, "w", encoding="utf-8") as f:
        json.dump({k: int(w[k]) for k in NAMES}, f, indent=2)


def to_vector(w):
    return np.array([w[k] for k in NAMES], dtype=float)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))


def mse(w, X, y, scale):
    return float(np.mean((sigmoid(X @ w / scale) - y) ** 2))


K_MIN, K_MAX = 20.0, 20000.0


def calibrate_scale(w, X, y):
    """가중치 w 를 고정하고, P(승) = sigmoid(점수 / K) 의 K 를 오차가 가장 작은 값으로 고른다."""
    grid = np.exp(np.linspace(np.log(K_MIN), np.log(K_MAX), 300))
    errs = [mse(w, X, y, k) for k in grid]
    return float(grid[int(np.argmin(errs))])


def load_samples(files):
    """files = [(라운드, jsonl 경로), ...]  ->  X(특징), y(결과), gid(판 번호; 라운드가 달라도 겹치지 않음)"""
    frames = []
    for rnd, path in files:
        df = pd.read_json(path, lines=True)
        df["gid"] = rnd * 1_000_000 + df["game"]
        frames.append(df)
    df = pd.concat(frames, ignore_index=True)
    X = np.array(df["features"].tolist(), dtype=float)
    if X.shape[1] != len(NAMES):
        raise ValueError(f"특징이 {X.shape[1]}개인데 NAMES 는 {len(NAMES)}개입니다. EvalWeights.java 와 맞춰 주세요.")
    keep = (df["score"].abs() < FORCED_SCORE).values
    print(f"국면 {len(df):,}개 중 확정 승패 {(~keep).sum():,}개 제외 -> {keep.sum():,}개 사용 "
          f"(판 {df['gid'].nunique()}개, 승 {np.mean(df['outcome'] == 1):.1%} / 무 {np.mean(df['outcome'] == 0.5):.1%} / 패 {np.mean(df['outcome'] == 0):.1%})")
    return (X[keep], df["outcome"].values[keep].astype(float), df["gid"].values[keep],
            df["score"].values[keep].astype(float))


def corr(a, b):
    if np.std(a) < 1e-9 or np.std(b) < 1e-9:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def diagnose(X, y, t, w0):
    """데이터에 학습할 만한 신호가 있는지 보여 준다. 상관계수가 0 에 가까우면 가중치를 바꿔도 얻을 게 적다."""
    ev = X @ w0
    print(f"신호 세기 (상관계수): 평가점수-승패 {corr(ev, y):+.3f} | 탐색점수-승패 {corr(t, y):+.3f} | 평가점수-탐색점수 {corr(ev, t):+.3f}")
    print(f"점수 표준편차: 평가점수 {ev.std():.0f} / 탐색점수 {t.std():.0f}  (평가가 훨씬 크면 평가가 탐색보다 과하게 확신한다는 뜻)")


def fit(X, y, w0, lam, scale):
    """제곱오차 + lam * |(w - w0) / REG_UNIT|^2 를 최소화하는 가중치. 눈금 scale 은 고정."""
    s = X.std(axis=0)
    s[s < 1e-9] = 1.0              # 상수 열 (tempo 등) 은 그대로 둔다
    Xs = X / s                     # 열 크기를 맞춘다

    # 변수 v = w * s / scale 로 바꿔서 기울기가 O(0.01~1) 이 되게 한다.
    # (w 그대로 쓰면 기울기가 1e-5 쯤이라 L-BFGS 가 시작하자마자 "수렴"으로 착각하고 멈춘다)
    def loss(v):
        w = v * scale / s
        p = sigmoid(Xs @ v)
        err = p - y
        diff = (w - w0) / REG_UNIT
        value = np.mean(err ** 2) + lam * np.sum(diff ** 2)
        grad = (2.0 / len(y)) * (Xs.T @ (err * p * (1.0 - p))) + 2.0 * lam * diff * scale / (REG_UNIT * s)
        return value, grad

    res = minimize(loss, w0 * s / scale, jac=True, method="L-BFGS-B",
                   options={"maxiter": 2000, "gtol": 1e-10, "ftol": 1e-14})
    return res.x * scale / s


def tune(X, y, gid, w0, seed=0):
    """눈금 K 를 보정하고 lam 을 검증 오차로 고른 뒤, 전체 데이터로 다시 맞춘 가중치(dict)를 돌려준다.
    신호가 너무 약해서 K 를 정할 수 없으면 None."""
    rng = np.random.default_rng(seed)
    games = np.unique(gid)
    rng.shuffle(games)
    val = np.isin(gid, games[: max(1, int(len(games) * 0.2))])
    tr = ~val

    k_train = calibrate_scale(w0, X[tr], y[tr])
    print(f"눈금 K = {k_train:.0f}  (현재 가중치 기준. 클수록 '점수가 승패를 덜 말해 준다'는 뜻)")
    if k_train > 0.8 * K_MAX:
        print("!! K 가 탐색 범위 끝에 붙었습니다. 현재 평가 점수가 자가대전 승패를 거의 설명하지 못한다는 뜻이라,")
        print("   가중치 크기가 임의로 정해져 학습 결과를 믿을 수 없습니다. 이 라운드는 학습을 건너뜁니다.")
        print("   (데이터는 쌓이므로 다음 라운드에 합쳐서 다시 시도합니다. GAMES_PER_ROUND / TIME_MS 를 늘리면 도움이 됩니다.)")
        return None
    print(f"검증 오차 기준선: 항상 0.5 예측 {np.mean((0.5 - y[val]) ** 2):.4f} | "
          f"현재 가중치 (K=200 가정) {mse(w0, X[val], y[val], 200.0):.4f} | 현재 가중치 (보정한 K) {mse(w0, X[val], y[val], k_train):.4f}")
    rows = []
    for lam in LAMBDAS:
        w = fit(X[tr], y[tr], w0, lam, k_train)
        rows.append({"lambda": lam, "train_mse": mse(w, X[tr], y[tr], k_train), "val_mse": mse(w, X[val], y[val], k_train)})
    table = pd.DataFrame(rows)
    print(table.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    best_lam = float(table.loc[table["val_mse"].idxmin(), "lambda"])
    print("선택한 lambda:", best_lam)

    k_all = calibrate_scale(w0, X, y)
    w = fit(X, y, w0, best_lam, k_all)
    w = np.clip(np.round(w), -MAX_ABS_WEIGHT, MAX_ABS_WEIGHT)
    return dict(zip(NAMES, w.astype(int).tolist()))

## 5. 학습 루프

라운드마다 현재 가중치로 자가대전 → (지금까지 모은 데이터 전체로) 학습 → 새 가중치 vs 현재 가중치 대국.
새 가중치의 점수가 `ACCEPT_SCORE` 이상이면 채택해서 다음 라운드의 "현재 가중치"로 씁니다.

출력 읽는 법:
- **신호 세기**: 평가점수-승패 상관이 0 에 가까우면 지금 평가 함수는 자가대전 승패를 거의 설명하지 못합니다. 이때는 학습해도 얻을 게 적습니다.
- **눈금 K 경고**: K 가 범위 끝에 붙으면 같은 뜻입니다.
- **대국 결과**: 95% 구간이 0.5 를 포함하면 아직 강해졌다고 단정할 수 없습니다. `MATCH_GAMES` 를 늘려 보세요. 이 게임은 무승부가 많아서 결판이 난 판이 적습니다.

In [ ]:
current = dict(DEFAULT_WEIGHTS)
current_path = WORK_DIR / "weights_round0.json"
write_weights(current_path, current)

data_files = []
log = []
for rnd in range(1, ROUNDS + 1):
    print(f"\n=============== 라운드 {rnd}/{ROUNDS} ===============")
    out = DATA_DIR / f"selfplay_round{rnd}.jsonl"
    print(f"[1/3] 자가대전 {GAMES_PER_ROUND}판 (현재 가중치)")
    run_selfplay(current_path, out, GAMES_PER_ROUND, SEED + rnd)
    data_files.append((rnd, out))

    print("[2/3] 학습")
    X, y, gid, t = load_samples(data_files)
    diagnose(X, y, t, to_vector(DEFAULT_WEIGHTS))
    candidate = tune(X, y, gid, to_vector(DEFAULT_WEIGHTS), seed=SEED)
    if candidate is None:
        log.append({"round": rnd, "positions": len(y), "A 승": None, "무": None, "B 승": None,
                    "A 점수": None, "95% 구간": "-", "채택": False})
        continue
    if candidate == current:
        print("-> 학습 결과가 현재 가중치와 같아서 대국을 건너뜁니다 (정규화가 강하거나 데이터가 부족)")
        log.append({"round": rnd, "positions": len(y), "A 승": None, "무": None, "B 승": None,
                    "A 점수": None, "95% 구간": "-", "채택": False})
        continue
    cand_path = WORK_DIR / f"weights_round{rnd}.json"
    write_weights(cand_path, candidate)

    print(f"[3/3] 대국 확인: 새 가중치(A) vs 현재 가중치(B), {MATCH_GAMES}판")
    m = run_match(cand_path, current_path, SEED + 1000 + rnd)
    accepted = m["score"] >= ACCEPT_SCORE
    log.append({"round": rnd, "positions": len(y), "A 승": m["aWins"], "무": m["draws"], "B 승": m["bWins"],
                "A 점수": m["score"], "95% 구간": f"{m['low']:.3f}~{m['high']:.3f}", "채택": accepted})
    if accepted:
        current, current_path = candidate, cand_path
        print("-> 채택")
    else:
        print("-> 현재 가중치보다 나아졌다고 보기 어려워 채택하지 않음")

print()
print(pd.DataFrame(log).to_string(index=False))

## 6. 결과: `weights.json`

In [ ]:
FINAL_PATH = WORK_DIR / "weights.json"
write_weights(FINAL_PATH, current)

summary = pd.DataFrame({"기본값": DEFAULT_WEIGHTS, "최종": current})
summary["변화"] = summary["최종"] - summary["기본값"]
print(summary.to_string())
print()
print("저장:", FINAL_PATH)
print("게임에 적용: 이 파일을 받아서  java -Dslider.weights=weights.json -cp target/classes org.example.Main")

try:
    from google.colab import files
    files.download(str(FINAL_PATH))
except ImportError:
    pass

### 결과를 게임에 쓰려면

```
java -Dslider.weights=weights.json -cp target/classes org.example.Main
```

- 가중치는 *수 선택 방식*이 아니라 *평가 함수*에만 영향을 줍니다. 규칙이나 탐색은 그대로입니다.
- 학습 데이터는 자가대전 결과라서, 처음 한두 라운드는 기본 가중치의 편향을 그대로 배웁니다. `ROUNDS` 를 늘려도 대국 점수가 오르지 않으면 평가 특징 자체(`SliderAI.computeFeatures`)에 새 항목을 추가하는 쪽이 효과적입니다.
- 가중치 이름과 특징 정의는 `EvalWeights.java` 의 주석을 보세요.